# MRI DICOM Protocol Fingerprints and Quality Control

A reusable metadata-first tool for MRI collections. It profiles acquisition protocols,
checks geometric consistency, exports compact QC tables, and renders representative
slice montages. No clinical labels or predictions are used.

In [ ]:
import hashlib
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from IPython.display import display

SEED = 20260806
HEADER_SAMPLE_SIZE = 4000
MONTAGE_SLICES = 9
MAX_PROTOCOL_MONTAGES = 6

def find_competition_root():
    candidates = [
        Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
        Path('/kaggle/input/rsna-knee-abnormality-detection'),
        Path('data/raw'),
    ]
    candidates += list(Path('/kaggle/input').glob('*knee*')) if Path('/kaggle/input').exists() else []
    for root in candidates:
        if (root / 'train.csv').exists() and (root / 'train_series.csv').exists():
            return root
    raise FileNotFoundError('Could not locate competition data')

ROOT = find_competition_root()
WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('outputs/dicom_protocol_qc')
WORK.mkdir(parents=True, exist_ok=True)
print({'data_root': str(ROOT), 'output_root': str(WORK)})

## Inventory

In [ ]:
studies = pd.read_csv(ROOT / 'train.csv', usecols=['StudyInstanceUID'])
series = pd.read_csv(ROOT / 'train_series.csv')

inventory = pd.DataFrame({
    'studies': [studies['StudyInstanceUID'].nunique()],
    'series': [len(series)],
    'series_per_study_median': [series.groupby('StudyInstanceUID').size().median()],
    'series_per_study_p95': [series.groupby('StudyInstanceUID').size().quantile(.95)],
})
display(inventory)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
series['Anatomical_Plane'].value_counts().plot.bar(ax=axes[0], color='#35689a', title='Series by anatomical plane')
axes[0].set_xlabel('')
axes[0].set_ylabel('series')
series.groupby('StudyInstanceUID').size().plot.hist(ax=axes[1], bins=range(2, 17), color='#57a773', title='Series per study')
axes[1].set_xlabel('number of series')
plt.tight_layout()

## Header sampling and protocol fingerprints

A first DICOM header is read from each sampled series with `stop_before_pixels=True`.
The fingerprint combines geometric and acquisition fields, making protocol variation
visible without loading the image volume.

In [ ]:
sample = series.sample(min(HEADER_SAMPLE_SIZE, len(series)), random_state=SEED).reset_index(drop=True)

def first_dicom_path(study_uid, series_uid):
    folder = ROOT / 'train_series' / study_uid / series_uid
    return next(folder.glob('*.dcm'), None)

def dcm_value(dataset, name, default=np.nan):
    value = dataset.get(name, default)
    if hasattr(value, 'value'):
        value = value.value
    if isinstance(value, (list, tuple)):
        return '\\'.join(map(str, value))
    return value

def normalize_number(value):
    try:
        return round(float(value), 4)
    except (TypeError, ValueError):
        return np.nan

def read_header(row):
    record = row.to_dict()
    path = first_dicom_path(row.StudyInstanceUID, row.SeriesInstanceUID)
    record['path_found'] = path is not None
    if path is None:
        record['header_error'] = 'missing_dicom'
        return record
    try:
        dataset = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        pixel_spacing = dcm_value(dataset, 'PixelSpacing', '')
        orientation = dcm_value(dataset, 'ImageOrientationPatient', '')
        record.update({
            'header_error': '',
            'Rows': normalize_number(dcm_value(dataset, 'Rows')),
            'Columns': normalize_number(dcm_value(dataset, 'Columns')),
            'PixelSpacing': str(pixel_spacing),
            'SliceThickness': normalize_number(dcm_value(dataset, 'SliceThickness')),
            'SpacingBetweenSlices': normalize_number(dcm_value(dataset, 'SpacingBetweenSlices')),
            'EchoTime': normalize_number(dcm_value(dataset, 'EchoTime')),
            'RepetitionTime': normalize_number(dcm_value(dataset, 'RepetitionTime')),
            'FlipAngle': normalize_number(dcm_value(dataset, 'FlipAngle')),
            'MagneticFieldStrength': normalize_number(dcm_value(dataset, 'MagneticFieldStrength')),
            'Manufacturer': str(dcm_value(dataset, 'Manufacturer', '')),
            'ManufacturerModelName': str(dcm_value(dataset, 'ManufacturerModelName', '')),
            'ImageOrientationPatient': str(orientation),
        })
    except Exception as exc:
        record['header_error'] = type(exc).__name__
    return record

with ThreadPoolExecutor(max_workers=min(16, os.cpu_count() or 4)) as executor:
    headers = list(executor.map(read_header, (row for _, row in sample.iterrows())))

qc = pd.DataFrame(headers)
for column in ['Rows', 'Columns', 'SliceThickness', 'SpacingBetweenSlices', 'EchoTime', 'RepetitionTime', 'FlipAngle', 'MagneticFieldStrength']:
    if column not in qc:
        qc[column] = np.nan

fingerprint_columns = [
    'Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression', 'Rows', 'Columns',
    'PixelSpacing', 'SliceThickness', 'SpacingBetweenSlices', 'EchoTime',
    'RepetitionTime', 'MagneticFieldStrength', 'Manufacturer',
]
fingerprint_text = qc[fingerprint_columns].fillna('NA').astype(str).agg('|'.join, axis=1)
qc['protocol_fingerprint'] = fingerprint_text.map(lambda text: hashlib.sha1(text.encode()).hexdigest()[:12])
qc['geometry_complete'] = qc[['Rows', 'Columns', 'PixelSpacing']].notna().all(axis=1)
qc['matrix_signature'] = qc['Rows'].fillna(-1).astype(int).astype(str) + 'x' + qc['Columns'].fillna(-1).astype(int).astype(str)
matrix_frequency = qc['matrix_signature'].value_counts()
qc['rare_matrix'] = qc['matrix_signature'].map(matrix_frequency).lt(10)
qc['qc_flag'] = np.select(
    [qc['header_error'].ne(''), ~qc['geometry_complete'], qc['rare_matrix']],
    ['header_read_error', 'missing_geometry', 'rare_matrix'],
    default='ok',
)

protocols = (
    qc.groupby(['protocol_fingerprint', *fingerprint_columns], dropna=False)
    .agg(series=('SeriesInstanceUID', 'size'), studies=('StudyInstanceUID', 'nunique'), qc_flags=('qc_flag', lambda x: int((x != 'ok').sum())))
    .reset_index()
    .sort_values('series', ascending=False)
)
qc.to_csv(WORK / 'dicom_series_qc.csv', index=False)
protocols.to_csv(WORK / 'protocol_fingerprints.csv', index=False)
display(protocols.head(12))

## Protocol diversity and QC flags

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
protocols.head(15).sort_values('series').plot.barh(x='protocol_fingerprint', y='series', legend=False, color='#5b8fb9', ax=axes[0])
axes[0].set_title('Most frequent protocol fingerprints')
axes[0].set_xlabel('sampled series')

qc['qc_flag'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1], colors=['#5bb98c', '#e4a64b', '#ce5d5d'], ylabel='')
axes[1].set_title('Header and geometry checks')
plt.tight_layout()

## Representative MRI montages

Three evenly distributed dimensions in a selected series are displayed as a 3×3 grid.
This makes orientation, field of view, and signal characteristics easy to compare across
common protocol fingerprints.

In [ ]:
def normalized_slice(path):
    image = pydicom.dcmread(path, force=True).pixel_array.astype(np.float32)
    low, high = np.percentile(image, [1, 99])
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)
    return np.clip((image - low) / (high - low), 0, 1)

def show_montage(row):
    folder = ROOT / 'train_series' / row.StudyInstanceUID / row.SeriesInstanceUID
    files = sorted(folder.glob('*.dcm'))
    if not files:
        return
    selection = np.linspace(0, len(files) - 1, MONTAGE_SLICES).round().astype(int)
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    for axis, index in zip(axes.flat, selection):
        try:
            axis.imshow(normalized_slice(files[index]), cmap='gray')
            axis.set_title(f'slice {index + 1}/{len(files)}', fontsize=8)
        except Exception as exc:
            axis.text(.5, .5, type(exc).__name__, ha='center', va='center', fontsize=8)
        axis.axis('off')
    fig.suptitle(
        f"{row.protocol_fingerprint} | {row.Anatomical_Plane} | fluid={row.Fluid_Sensitive} | fat-sat={row.Fat_Suppression}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

representatives = (
    qc[qc['qc_flag'].eq('ok')]
    .sort_values('protocol_fingerprint')
    .groupby('protocol_fingerprint', as_index=False)
    .first()
    .merge(protocols[['protocol_fingerprint', 'series']], on='protocol_fingerprint')
    .sort_values('series', ascending=False)
    .head(MAX_PROTOCOL_MONTAGES)
)
for _, representative in representatives.iterrows():
    show_montage(representative)

## Output files

- `dicom_series_qc.csv`: one sampled series per row, including decoded header fields and QC flag.
- `protocol_fingerprints.csv`: grouped protocol signatures with sampled series and study counts.

In [ ]:
print({
    'sampled_series': len(qc),
    'unique_protocols': qc['protocol_fingerprint'].nunique(),
    'qc_flagged': int((qc['qc_flag'] != 'ok').sum()),
    'series_qc_csv': str(WORK / 'dicom_series_qc.csv'),
    'protocol_csv': str(WORK / 'protocol_fingerprints.csv'),
})